<a href="https://colab.research.google.com/github/poornimasampathkumarjam-netizen/customer-churn-analysis/blob/main/customer_churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import classification_report,confusion_matrix, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE



**LOAD AND PREPARE THE DATASET**

In [64]:
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [65]:
df.shape

(7043, 21)

In [66]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [68]:
df = df.drop(columns=["customerID"])

In [60]:
df.head(2)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,0


In [69]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [70]:
num_feature_list = ["tenure","MonthlyCharges", "TotalCharges"]
for col in df.columns:
  if col not in num_feature_list:
     print(col, df[col].unique())
     print("-"*70)


gender ['Female' 'Male']
----------------------------------------------------------------------
SeniorCitizen [0 1]
----------------------------------------------------------------------
Partner ['Yes' 'No']
----------------------------------------------------------------------
Dependents ['No' 'Yes']
----------------------------------------------------------------------
PhoneService ['No' 'Yes']
----------------------------------------------------------------------
MultipleLines ['No phone service' 'No' 'Yes']
----------------------------------------------------------------------
InternetService ['DSL' 'Fiber optic' 'No']
----------------------------------------------------------------------
OnlineSecurity ['No' 'Yes' 'No internet service']
----------------------------------------------------------------------
OnlineBackup ['Yes' 'No' 'No internet service']
----------------------------------------------------------------------
DeviceProtection ['No' 'Yes' 'No internet service']
------

In [71]:
df.isnull().sum()

,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0
OnlineBackup,0


In [72]:
(df.isnull().sum() / len(df)) * 100

,0
gender,0.0
SeniorCitizen,0.0
Partner,0.0
Dependents,0.0
tenure,0.0
PhoneService,0.0
MultipleLines,0.0
InternetService,0.0
OnlineSecurity,0.0
OnlineBackup,0.0


In [73]:
df.duplicated().sum()

np.int64(22)

In [74]:
df[df["TotalCharges"] ==" "]



,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


In [75]:
len(df[df["TotalCharges"] ==" "])

11

In [76]:
df["TotalCharges"] = df["TotalCharges"].replace({" " : "0.0"})

In [77]:
df["TotalCharges"] = df["TotalCharges"].astype(float)
print(df.TotalCharges)

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: TotalCharges, Length: 7043, dtype: float64


**ONE HOT ENCODING**

In [78]:
df["Churn"] = df["Churn"].replace({"Yes": 1,"No": 0})


/tmp/ipykernel_3795/2416429115.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Churn"] = df["Churn"].replace({"Yes": 1,"No": 0})


In [79]:
df.head(2)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0


In [80]:
print(df["Churn"].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [83]:
X = df.drop(columns=["Churn"])
Y = df["Churn"]
categorical_columns = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
X = pd.get_dummies(X, columns = categorical_columns, drop_first=True)
print(X,Y)

      MonthlyCharges  TotalCharges  gender_Male  SeniorCitizen_1  Partner_Yes  \
0              29.85         29.85        False            False         True   
1              56.95       1889.50         True            False        False   
2              53.85        108.15         True            False        False   
3              42.30       1840.75         True            False        False   
4              70.70        151.65        False            False        False   
...              ...           ...          ...              ...          ...   
7038           84.80       1990.50         True            False         True   
7039          103.20       7362.90        False            False         True   
7040           29.60        346.45        False            False         True   
7041           74.40        306.60         True             True         True   
7042          105.65       6844.50         True            False        False   

      Dependents_Yes  tenur

**TRAIN TEST SPLIT**

In [91]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size = 0.2, random_state = 42, stratify = Y)
scaler =StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns = X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns = X_test.columns)
print(X_train.shape,X_test.shape)
print(Y_train.value_counts())

(5634, 101) (1409, 101)
Churn
0    4139
1    1495
Name: count, dtype: int64


**SMOTE**

In [92]:
smote = SMOTE(random_state = 42)
X_train_smote,Y_train_smote = smote.fit_resample(X_train, Y_train)
print(X_train.shape,Y_train.shape)
print(Y_train_smote.value_counts())


(5634, 101) (5634,)
Churn
0    4139
1    4139
Name: count, dtype: int64


**MODEL TRAINING**

In [96]:
models = {
    "DT": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42)
    }

In [106]:
model = models["RF"]
model.fit(X_train_smote, Y_train_smote)
Y_test_pred = model.predict(X_test)
print("accuracy", accuracy_score(Y_test, Y_test_pred))
print("confusion_matrix", confusion_matrix(Y_test, Y_test_pred))
print("classification_report", classification_report(Y_test, Y_test_pred))

accuracy 0.7792760823278921
confusion_matrix [[907 128]
 [183 191]]
classification_report               precision    recall  f1-score   support

           0       0.83      0.88      0.85      1035
           1       0.60      0.51      0.55       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409



In [107]:
model = models["DT"]
model.fit(X_train_smote, Y_train_smote)
Y_test_pred = model.predict(X_test)
print("accuracy", accuracy_score(Y_test, Y_test_pred))
print("confusion_matrix", confusion_matrix(Y_test, Y_test_pred))
print("classification_report", classification_report(Y_test, Y_test_pred))

accuracy 0.7388218594748048
confusion_matrix [[842 193]
 [175 199]]
classification_report               precision    recall  f1-score   support

           0       0.83      0.81      0.82      1035
           1       0.51      0.53      0.52       374

    accuracy                           0.74      1409
   macro avg       0.67      0.67      0.67      1409
weighted avg       0.74      0.74      0.74      1409



**CHURN PROBBILITY**

In [109]:
Y_test_probability = model.predict_proba(X_test)[:,1]
probability_df= pd.DataFrame({
    "actual_churn": Y_test.values,
    "predicted_churn": Y_test_pred,
    "churn_probability": Y_test_probability
})
print(probability_df.head(10))

   actual_churn  predicted_churn  churn_probability
0             0                0                0.0
1             0                0                0.0
2             0                0                0.0
3             0                1                1.0
4             0                0                0.0
5             0                0                0.0
6             0                1                1.0
7             0                1                1.0
8             0                0                0.0
9             1                1                1.0
